# Cross-Dataset Analysis — All 4 Benchmarks

Reads results from Google Drive for HotpotQA, FEVER, MuSiQue, and 2WikiMultiHopQA.
Produces descriptive cross-dataset comparison tables and figures.

**Audit status:** historical FEVER evidence and token budgets require validation. Best-unc is a retrospective maximum that may use judge-informed hints. Saved outputs were cleared after correcting mixed seed aggregation; rerun with the original raw logs. See `docs/iclr2027_readiness.md`.

**No GPU required** — runs on any Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/kishormorol/agent-repair.git /content/agent-repair 2>/dev/null || (cd /content/agent-repair && git pull)
!pip install -q scipy scikit-learn pandas pyarrow pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

import os
os.chdir('/content/agent-repair')
print('Ready')

In [ ]:
import os, json, glob
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 11,
    'figure.facecolor': 'white'
})

# Drive paths for each dataset
DRIVE_BASES = {
    'HotpotQA': '/content/drive/MyDrive/agent-repair-hotpotqa',
    'FEVER': '/content/drive/MyDrive/agent-repair-fever',
    'MuSiQue': '/content/drive/MyDrive/agent-repair-musique',
    '2WikiMHQA': '/content/drive/MyDrive/agent-repair-2wikimultihopqa',
}

# Output directory for cross-dataset figures and tables
CROSS_OUT = '/content/drive/MyDrive/agent-repair-cross'
os.makedirs(CROSS_OUT, exist_ok=True)

# Verify all paths exist
for name, path in DRIVE_BASES.items():
    exists = os.path.exists(path)
    print(f'{name:12s}: {"OK" if exists else "MISSING"} — {path}')

---
## 1. Load Results from All Datasets

In [ ]:
import sys
sys.path.insert(0, '/content/agent-repair')
from src.utils import read_jsonl
from src.repair import parse_strategy
from src.eval import summarize_strategies, ensemble_rows

all_results = {}
all_localization = {}
all_readable = {}

for ds_name, base in DRIVE_BASES.items():
    # Load repair results
    results_path = os.path.join(base, 'outputs/repairs/results.jsonl')
    if os.path.exists(results_path):
        df = pd.DataFrame(list(read_jsonl(results_path)))
        if 'multiplier' not in df.columns:
            df['multiplier'] = 1.0
        if 'backtrack' not in df.columns:
            df['backtrack'] = df['strategy'].apply(lambda s: parse_strategy(s)['backtrack'])
        df = df[df.multiplier == 1.0].copy()
        df = df.drop_duplicates(subset=['qid', 'strategy', 'seed'])
        df['dataset'] = ds_name
        all_results[ds_name] = df
        print(f'{ds_name:12s}: {len(df):>6,} repair rows')
    else:
        print(f'{ds_name:12s}: results.jsonl NOT FOUND')

    # Load localization summary
    loc_path = os.path.join(base, 'outputs/tables/localization_summary.csv')
    if os.path.exists(loc_path):
        loc_df = pd.read_csv(loc_path)
        loc_df['dataset'] = ds_name
        all_localization[ds_name] = loc_df

    # Load readable results
    readable_path = os.path.join(base, 'outputs/tables/main_results_readable.csv')
    if os.path.exists(readable_path):
        all_readable[ds_name] = pd.read_csv(readable_path)

print(f'\nLoaded {len(all_results)} datasets')

In [ ]:
# Combine all repair results into one DataFrame
combined = pd.concat(all_results.values(), ignore_index=True)
print(f'Combined: {len(combined):,} rows across {combined.dataset.nunique()} datasets')
print(f'Strategies: {combined.strategy.nunique()}')
print(f'\nFailed trajectories per dataset:')
for ds in combined.dataset.unique():
    n = combined[combined.dataset == ds].qid.nunique()
    print(f'  {ds:12s}: {n}')

---
## 2. Cross-Dataset Main Results Table

In [ ]:
# Key strategies to compare
KEY_STRATEGIES = [
    'random_step', 'oracle_targeted', 'oracle_targeted__bt2',
    'oracle_targeted__bt2__informed', 'oracle_targeted__informed',
    'full_restart',
]

# Build cross-dataset comparison table
rows = []
for ds_name, df in all_results.items():
    for strat in KEY_STRATEGIES:
        sub = df[df.strategy == strat]
        if sub.empty:
            continue
        # Mean seed success per question for every strategy category.
        mv = sub.groupby('qid')['success'].mean().mean()
        rows.append({'dataset': ds_name, 'strategy': strat, 'fix_rate': mv})

    # Retrospective maximum, including judge-informed variants.
    unc = df[df.strategy.str.startswith('unc__')]
    if not unc.empty:
        rates = unc.groupby(['strategy', 'qid'])['success'].mean().groupby('strategy').mean()
        best_name = rates.idxmax()
        best_rate = rates.max()
        rows.append({'dataset': ds_name, 'strategy': 'best_unc', 'fix_rate': best_rate,
                     'best_unc_name': best_name})

    # Ensemble
    ens_sub = df[df.strategy == 'uncertainty_ensemble_any']
    if ens_sub.empty:
        # Compute ensemble
        bt0_unc = [s for s in unc.strategy.unique() if parse_strategy(s)['backtrack'] == 0]
        ens_rows = ensemble_rows(df, bt0_unc, 'uncertainty_ensemble_any')
        ens_rate = ens_rows.groupby('qid')['success'].mean().mean() if len(ens_rows) > 0 else 0
    else:
        ens_rate = ens_sub.groupby('qid')['success'].mean().mean()
    rows.append({'dataset': ds_name, 'strategy': 'ensemble', 'fix_rate': ens_rate})

cross = pd.DataFrame(rows)

# Pivot table
pivot = cross.pivot_table(index='strategy', columns='dataset', values='fix_rate')
# Reorder
strat_order = KEY_STRATEGIES + ['best_unc', 'ensemble']
ds_order = ['HotpotQA', 'FEVER', 'MuSiQue', '2WikiMHQA']
pivot = pivot.reindex(index=[s for s in strat_order if s in pivot.index],
                      columns=[d for d in ds_order if d in pivot.columns])
pivot_pct = (pivot * 100).round(1)
print('Cross-Dataset Repair Success Rates (%)')
print('=' * 70)
display(pivot_pct)

---
## 3. Cross-Dataset Localization Comparison

In [ ]:
# Combine localization results
if all_localization:
    loc_combined = pd.concat(all_localization.values(), ignore_index=True)

    # Key metrics only
    key_metrics = ['self_consistency', 'token_entropy_max', 'max_token_prob_max',
                   'perplexity', 'verbalized_confidence']
    loc_key = loc_combined[loc_combined.metric.isin(key_metrics)]

    loc_pivot = loc_key.pivot_table(index='metric', columns='dataset',
                                    values='argmax_top1')
    loc_pivot = loc_pivot.reindex(
        index=[m for m in key_metrics if m in loc_pivot.index],
        columns=[d for d in ds_order if d in loc_pivot.columns])
    print('Localization Accuracy (argmax top-1) by Metric and Dataset')
    print('=' * 70)
    display(loc_pivot.round(3))
else:
    print('No localization data found')

---
## 4. Figure: Cross-Dataset Strategy Comparison (Grouped Bar Chart)

In [ ]:
# Grouped bar chart: key strategies across datasets
plot_strats = ['random_step', 'oracle_targeted', 'oracle_targeted__bt2',
               'full_restart', 'best_unc', 'ensemble']
plot_labels = ['Random', 'Oracle', 'Oracle+bt2', 'Full Restart',
               'Best Unc.', 'Ensemble']
plot_colors = ['#999999', '#009E73', '#56B4E9', '#0072B2', '#E69F00', '#D55E00']

fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(ds_order))
n_strats = len(plot_strats)
width = 0.8 / n_strats

for i, (strat, label, color) in enumerate(zip(plot_strats, plot_labels, plot_colors)):
    vals = []
    for ds in ds_order:
        row = cross[(cross.dataset == ds) & (cross.strategy == strat)]
        vals.append(row['fix_rate'].values[0] * 100 if len(row) > 0 else 0)
    offset = (i - n_strats / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=label, color=color,
                  edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(ds_order, fontsize=12)
ax.set_ylabel('Fix Rate (%)', fontsize=12)
ax.set_title('Repair Success Across Datasets', fontsize=14)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.set_ylim(0, max(cross.fix_rate) * 110)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_cross_dataset_bars.png', dpi=300)
plt.show()
print('Saved fig_cross_dataset_bars.png')

---
## 5. Figure: Backtracking Effect Across Datasets

In [ ]:
# Backtracking effect: oracle vs oracle+bt2 vs full_restart per dataset
bt_strats = ['oracle_targeted', 'oracle_targeted__bt2', 'full_restart']
bt_labels = ['Oracle (exact step)', 'Oracle + bt2', 'Full Restart']
bt_colors = ['#009E73', '#56B4E9', '#0072B2']

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(ds_order))
width = 0.25

for i, (strat, label, color) in enumerate(zip(bt_strats, bt_labels, bt_colors)):
    vals = []
    for ds in ds_order:
        row = cross[(cross.dataset == ds) & (cross.strategy == strat)]
        vals.append(row['fix_rate'].values[0] * 100 if len(row) > 0 else 0)
    bars = ax.bar(x + (i - 1) * width, vals, width, label=label, color=color,
                  edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(ds_order, fontsize=12)
ax.set_ylabel('Fix Rate (%)', fontsize=12)
ax.set_title('Backtracking Effect: Oracle vs Full Restart', fontsize=14)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_backtracking_effect.png', dpi=300)
plt.show()
print('Saved fig_backtracking_effect.png')

---
## 6. Figure: Localization Heatmap Across Datasets

In [ ]:
if all_localization:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    metric_labels = {
        'self_consistency': 'Self-Consistency',
        'token_entropy_max': 'Entropy (max)',
        'max_token_prob_max': 'MaxProb (max)',
        'perplexity': 'Perplexity',
        'verbalized_confidence': 'Verbalized Conf.'
    }
    data = loc_pivot.rename(index=metric_labels)
    im = ax.imshow(data.values, cmap='YlGn', aspect='auto',
                   vmin=0, vmax=max(0.01, float(np.nanmax(data.values))))
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels(data.columns, fontsize=11)
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index, fontsize=11)
    for i in range(len(data.index)):
        for j in range(len(data.columns)):
            v = data.values[i, j]
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=10,
                    color='white' if v > 0.3 else 'black')
    ax.set_title('Localization Accuracy (top-1) by Metric and Dataset', fontsize=13)
    fig.colorbar(im, ax=ax, label='Top-1 accuracy', shrink=0.8)
    fig.tight_layout()
    fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_localization_heatmap.png', dpi=300)
    plt.show()
    print('Saved fig_localization_heatmap.png')

---
## 7. Figure: Metric × Rule Heatmap (Averaged Across Datasets)

In [ ]:
# Average repair success by metric x rule across all datasets (bt0 only)
unc_all = combined[combined.strategy.str.startswith('unc__')].copy()
unc_all['metric'] = unc_all['strategy'].apply(lambda s: parse_strategy(s)['metric'])
unc_all['rule'] = unc_all['strategy'].apply(lambda s: parse_strategy(s)['rule'])
unc_all['bt'] = unc_all['strategy'].apply(lambda s: parse_strategy(s)['backtrack'])
unc_all['informed'] = unc_all['strategy'].apply(lambda s: parse_strategy(s)['informed'])

# bt0 generic only for clean comparison
unc_bt0 = unc_all[(unc_all.bt == 0) & (~unc_all.informed)]

piv = unc_bt0.groupby(['metric', 'rule'])['success'].mean().unstack('rule')

metric_order = ['self_consistency', 'token_entropy_max', 'max_token_prob_max',
                'perplexity', 'verbalized_confidence']
rule_order = ['argmax', 'topk', 'earliest_above_threshold',
              'cascade_upstream', 'cascade_gradient', 'cascade_weighted']
metric_labels = {'self_consistency': 'Self-Consist.', 'token_entropy_max': 'Entropy',
                 'max_token_prob_max': 'MaxProb', 'perplexity': 'Perplexity',
                 'verbalized_confidence': 'Verbalized'}
rule_labels = {'argmax': 'argmax', 'topk': 'top-k',
               'earliest_above_threshold': 'earliest>thr',
               'cascade_upstream': 'cascade-up',
               'cascade_gradient': 'cascade-grad',
               'cascade_weighted': 'cascade-wt'}

mo = [m for m in metric_order if m in piv.index]
ro = [r for r in rule_order if r in piv.columns]
piv = piv.loc[mo, ro]

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(piv.values, cmap='YlGn', aspect='auto',
               vmin=0, vmax=max(0.01, float(np.nanmax(piv.values))))
ax.set_xticks(range(len(ro)))
ax.set_xticklabels([rule_labels.get(r, r) for r in ro], fontsize=10)
ax.set_yticks(range(len(mo)))
ax.set_yticklabels([metric_labels.get(m, m) for m in mo], fontsize=10)
for i in range(len(mo)):
    for j in range(len(ro)):
        v = piv.values[i, j]
        ax.text(j, i, f'{100*v:.1f}', ha='center', va='center', fontsize=9,
                color='white' if v > 0.15 else 'black')
ax.set_title('Repair Success (%) by Uncertainty Metric × Rule\n(bt0, generic nudge, averaged across 4 datasets)', fontsize=12)
fig.colorbar(im, ax=ax, label='Success rate', shrink=0.8)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_metric_rule_heatmap_cross.png', dpi=300)
plt.show()
print('Saved fig_metric_rule_heatmap_cross.png')

---
## 8. Figure: Trajectory Length vs Repair Strategy Effectiveness

In [ ]:
# For each dataset: avg tool calls (from full_restart) vs fix rates
traj_stats = []
for ds_name, df in all_results.items():
    fr = df[df.strategy == 'full_restart']
    avg_tools = fr['recovery_tool_calls'].mean() if 'recovery_tool_calls' in fr.columns else 0

    oracle_rate = df[df.strategy == 'oracle_targeted']['success'].mean()
    oracle_bt2_rate = df[df.strategy == 'oracle_targeted__bt2']['success'].mean()
    restart_rate = df[df.strategy == 'full_restart']['success'].mean()

    unc = df[df.strategy.str.startswith('unc__')]
    best_unc_rate = unc.groupby('strategy')['success'].mean().max() if not unc.empty else 0

    traj_stats.append({
        'dataset': ds_name, 'avg_tool_calls': avg_tools,
        'oracle': oracle_rate, 'oracle_bt2': oracle_bt2_rate,
        'full_restart': restart_rate, 'best_unc': best_unc_rate,
        'n_failed': df.qid.nunique(),
    })

traj_df = pd.DataFrame(traj_stats)
display(traj_df[['dataset', 'avg_tool_calls', 'n_failed', 'oracle', 'oracle_bt2',
                 'full_restart', 'best_unc']].round(3))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for col, label, color, marker in [
    ('oracle', 'Oracle', '#009E73', 'o'),
    ('oracle_bt2', 'Oracle+bt2', '#56B4E9', 's'),
    ('full_restart', 'Full Restart', '#0072B2', 'D'),
    ('best_unc', 'Best Unc.', '#E69F00', '^'),
]:
    ax.plot(traj_df['avg_tool_calls'], traj_df[col] * 100, marker=marker,
            label=label, color=color, linewidth=0, markersize=10)
    for _, row in traj_df.iterrows():
        ax.annotate(row['dataset'], (row['avg_tool_calls'], row[col] * 100),
                    textcoords='offset points', xytext=(8, 0), fontsize=8,
                    color=color)

ax.set_xlabel('Avg tool calls in full-restart repairs', fontsize=12)
ax.set_ylabel('Fix Rate (%)', fontsize=12)
ax.set_title('Restart Tool Use vs Repair Effectiveness (descriptive)', fontsize=14)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_traj_length_vs_repair.png', dpi=300)
plt.show()
print('Saved fig_traj_length_vs_repair.png')

---
## 9. Figure: Cost-Accuracy Pareto (All Datasets)

In [ ]:
fig, axes = plt.subplots(1, len(ds_order), figsize=(4.5 * len(ds_order), 4.5))
if len(ds_order) == 1:
    axes = [axes]

base_colors = {'random_step': '#999999', 'full_restart': '#0072B2',
               'oracle_targeted': '#009E73', 'uncertainty_ensemble_any': '#D55E00'}
base_labels = {'random_step': 'Random', 'full_restart': 'Restart',
               'oracle_targeted': 'Oracle', 'uncertainty_ensemble_any': 'Ensemble'}

for idx, ds_name in enumerate(ds_order):
    if ds_name not in all_results:
        continue
    ax = axes[idx]
    df = all_results[ds_name]

    strat_agg = df.groupby('strategy').agg(
        success=('success', 'mean'),
        cost=('recovery_gen_tokens', 'mean')).reset_index()

    # Uncertainty strategies as grey dots (exclude ensemble)
    unc = strat_agg[(strat_agg.strategy.str.startswith('unc__')) &
                    (~strat_agg.strategy.str.contains('ensemble'))]
    ax.scatter(unc.cost, unc.success * 100, s=20, color='#b0b8c0', zorder=1, alpha=0.6)

    # Baselines as colored dots
    for s, c in base_colors.items():
        row = strat_agg[strat_agg.strategy == s]
        if not row.empty:
            ax.scatter(row.cost.values[0], row.success.values[0] * 100,
                       s=90, color=c, zorder=3, edgecolor='white', linewidth=1.5)
            ax.annotate(base_labels[s],
                        (row.cost.values[0], row.success.values[0] * 100),
                        textcoords='offset points', xytext=(7, 5),
                        fontsize=8, color=c, fontweight='bold')

    ax.set_xlabel('Avg Tokens')
    ax.set_ylabel('Fix Rate (%)')
    ax.set_title(ds_name, fontsize=12)

fig.suptitle('Cost vs Success: All Datasets', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_pareto_all_datasets.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig_pareto_all_datasets.png')

---
## 10. Figure: Backtracking Gain by Dataset

In [ ]:
# Show the backtracking gain (oracle vs oracle+bt2) per dataset
bt_gains = []
for ds_name, df in all_results.items():
    oracle = df[df.strategy == 'oracle_targeted']['success'].mean()
    oracle_bt2 = df[df.strategy == 'oracle_targeted__bt2']['success'].mean()
    restart = df[df.strategy == 'full_restart']['success'].mean()
    bt_gains.append({
        'dataset': ds_name,
        'oracle': oracle * 100,
        'oracle_bt2': oracle_bt2 * 100,
        'bt_gain': (oracle_bt2 - oracle) * 100,
        'full_restart': restart * 100,
    })
bt_gain_df = pd.DataFrame(bt_gains)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: absolute rates
x = np.arange(len(bt_gain_df))
w = 0.25
ax1.bar(x - w, bt_gain_df['oracle'], w, label='Oracle', color='#009E73', edgecolor='white')
ax1.bar(x, bt_gain_df['oracle_bt2'], w, label='Oracle+bt2', color='#56B4E9', edgecolor='white')
ax1.bar(x + w, bt_gain_df['full_restart'], w, label='Full Restart', color='#0072B2', edgecolor='white')
ax1.set_xticks(x)
ax1.set_xticklabels(bt_gain_df['dataset'])
ax1.set_ylabel('Fix Rate (%)')
ax1.set_title('Oracle vs Oracle+bt2 vs Full Restart')
ax1.legend(fontsize=9)

# Right: backtracking gain
colors = ['#D55E00' if g > 5 else '#E69F00' if g > 1 else '#999999'
          for g in bt_gain_df['bt_gain']]
ax2.bar(x, bt_gain_df['bt_gain'], color=colors, edgecolor='white')
for xi, g in zip(x, bt_gain_df['bt_gain']):
    ax2.text(xi, g + 0.3, f'+{g:.1f}', ha='center', fontsize=10, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(bt_gain_df['dataset'])
ax2.set_ylabel('Gain from Backtracking (pp)')
ax2.set_title('Backtracking Gain: Oracle+bt2 − Oracle')

fig.tight_layout()
fig.savefig('/content/drive/MyDrive/agent-repair-cross/fig_backtracking_gain.png', dpi=300)
plt.show()
print('Saved fig_backtracking_gain.png')

---
## 11. Summary Statistics and LaTeX Tables

In [ ]:
# Generate LaTeX for Table 1 (main results)
print('% LaTeX: Main Results Table (for paper)')
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{Repair success rates (\%) across four benchmarks.}')
print(r'\label{tab:main_cross}')
print(r'\small')
print(r'\begin{tabular}{@{}lcccc@{}}')
print(r'\toprule')
print(r'\textbf{Strategy} & \textbf{HotpotQA} & \textbf{FEVER} & \textbf{MuSiQue} & \textbf{2Wiki} \\')
print(r'\midrule')

strat_labels = {
    'random_step': 'Random step',
    'oracle_targeted': 'Oracle targeted',
    'oracle_targeted__bt2': 'Oracle + bt2',
    'full_restart': 'Full restart',
    'best_unc': 'Retrospective best unc. (all variants)',
    'ensemble': 'Ensemble (any)',
}

for strat in ['random_step', 'oracle_targeted', 'oracle_targeted__bt2',
              'full_restart', 'best_unc']:
    label = strat_labels[strat]
    vals = []
    for ds in ds_order:
        row = cross[(cross.dataset == ds) & (cross.strategy == strat)]
        v = f'{row["fix_rate"].values[0] * 100:.1f}' if len(row) > 0 else '--'
        vals.append(v)
    print(f'{label} & {" & ".join(vals)} \\\\')

print(r'\midrule')
vals = []
for ds in ds_order:
    row = cross[(cross.dataset == ds) & (cross.strategy == 'ensemble')]
    v = f'{row["fix_rate"].values[0] * 100:.1f}' if len(row) > 0 else '--'
    vals.append(v)
print(f'Ensemble (any) & {" & ".join(vals)} \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

In [ ]:
# Cross-dataset summary statistics
print('=' * 70)
print('CROSS-DATASET SUMMARY')
print('=' * 70)

for ds_name in ds_order:
    if ds_name not in all_results:
        continue
    df = all_results[ds_name]
    n_failed = df.qid.nunique()
    fr = df[df.strategy == 'full_restart']
    avg_tools = fr['recovery_tool_calls'].mean() if 'recovery_tool_calls' in fr.columns else 0

    oracle = df[df.strategy == 'oracle_targeted']['success'].mean() * 100
    oracle_bt2 = df[df.strategy == 'oracle_targeted__bt2']['success'].mean() * 100
    restart = df[df.strategy == 'full_restart']['success'].mean() * 100
    random = df[df.strategy == 'random_step']['success'].mean() * 100

    unc = df[df.strategy.str.startswith('unc__')]
    best_unc_rate = unc.groupby('strategy')['success'].mean().max() * 100 if not unc.empty else 0
    best_unc_name = unc.groupby('strategy')['success'].mean().idxmax() if not unc.empty else '--'

    winner = 'Full Restart' if restart > oracle_bt2 and restart > best_unc_rate else (
        'Best Unc.' if best_unc_rate > restart else 'Oracle+bt2')

    print(f'\n{ds_name} ({n_failed} failed, avg {avg_tools:.1f} tool calls):')
    print(f'  Random:      {random:.1f}%')
    print(f'  Oracle:      {oracle:.1f}%  →  Oracle+bt2: {oracle_bt2:.1f}%  (bt gain: +{oracle_bt2-oracle:.1f})')
    print(f'  Full Restart:{restart:.1f}%')
    print(f'  Best Unc.:   {best_unc_rate:.1f}%  ({best_unc_name})')
    print(f'  → Winner: {winner}')

print('\n' + '=' * 70)
print('KEY TAKEAWAYS:')
print('  Descriptive summaries only; best-unc is selected on the evaluated questions.')
print('  Best-unc and ensemble may include judge-informed variants.')
print('  Ensemble success uses gold answers: it is candidate coverage, not answer selection.')
print('  Restart tool calls are outcomes, not original trajectory lengths.')
print('  FEVER evidence and historical token budgets require audit before publication.')
print('=' * 70)

In [ ]:
# Save all cross-dataset outputs
import os
out_dir = '/content/drive/MyDrive/agent-repair-cross'
os.makedirs(out_dir, exist_ok=True)

pivot_pct.to_csv(os.path.join(out_dir, 'cross_dataset_main_results.csv'))
if all_localization:
    loc_pivot.to_csv(os.path.join(out_dir, 'cross_dataset_localization.csv'))
traj_df.to_csv(os.path.join(out_dir, 'cross_dataset_traj_stats.csv'), index=False)
bt_gain_df.to_csv(os.path.join(out_dir, 'cross_dataset_backtrack_gains.csv'), index=False)

print(f'All cross-dataset outputs saved to {out_dir}')
!ls -la {out_dir}